# 4. SQL Analysis

Eleven analytical queries on the enriched dataset answer business questions about Israeli financial apps over time. Full queries in `sql/02_analysis.sql`, schema in `sql/01_schema.sql`. The header of `02_analysis.sql` contains a 20-line SUMMARY OF FINDINGS for quick reading.

This notebook surfaces the key findings, the query catalog, and one demo query to show the SQL style.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 50)

import sqlite3
from pathlib import Path

# SQLite connection. The repo ships with the populated DB.
DB_PATH = Path("..") / "data" / "processed" / "reviews.db"
engine = sqlite3.connect(DB_PATH)

## 4.1 Key Findings

- All 11 apps declined from their historical peak years.
- Four foundational topics drive negative scores: stability (78% negative), security (75%), login_auth (67%), performance (63%). Features and fees are polarized.
- Each bank has a distinct complaint personality. Bank Leumi and Isracard lead on stability (35% each). One Zero on customer service (24%). Mizrahi-Tefahot on features and UI.
- FIBI recovery from 2.25 (2023) to 4.42 (2026) coincides with stability share falling 14 points and praise rising 29.
- Pepper decline from 4.50 (2021) to 1.95 (2025) coincides with UI/UX share rising 24 points and praise dropping 50. Consistent with a redesign that hurt users.
- An Isracard April 2023 incident (162 reviews at 1.47 avg, triple normal volume) traces to a failed update, visible in monthly drill-down and 30 sampled negative reviews.

## 4.2 Query Catalog

| # | Query | Question |
|---|---|---|
| 1 | App overview snapshot | How does each app perform in absolute terms? |
| 2 | Annual trajectory per app | How does score evolve year over year? |
| 3 | Peak vs recent decline | Who fell the most from their best year? |
| 4 | Segment YoY with LAG | Do segments move together or separately? |
| 5 | Monthly drill-down 2023-2024 | Where did each segment hit trouble? |
| 6 | Isracard incident: monthly per credit-card app | Did one app drive the credit-card crash? |
| 7 | Sample of 30 negative reviews | What did angry users write? |
| 8 | Topic-level score drivers | Which topics correlate with low scores? |
| 9 | Topic share per app | What is each bank's complaint profile? |
| 10 | FIBI recovery: topic mix shift | What did FIBI fix between 2023 and 2026? |
| 11 | Pepper decline: topic mix shift | What broke between 2021 and 2025? |

## 4.3 Demo Query: Peak vs Recent

Query 3 from the catalog. Identifies how far each app fell from its best year. CTE aggregates yearly scores (minimum 30 reviews to reduce noise), then computes peak versus recent (2025+) average.

In [2]:
peak_vs_recent = pd.read_sql("""
WITH yearly AS (
    SELECT 
        a.name, a.segment,
        CAST(strftime('%Y', r.review_date) AS INTEGER) AS yr,
        AVG(CAST(r.score AS REAL)) AS yr_score,
        COUNT(*) AS n_reviews
    FROM apps a
    INNER JOIN reviews r ON a.app_id = r.app_id
    WHERE r.review_date IS NOT NULL
    GROUP BY a.name, a.segment, CAST(strftime('%Y', r.review_date) AS INTEGER)
    HAVING COUNT(*) >= 30
)
SELECT 
    name, segment,
    ROUND(MAX(yr_score), 2) AS peak,
    ROUND(AVG(CASE WHEN yr >= 2025 THEN yr_score END), 2) AS recent,
    ROUND(AVG(CASE WHEN yr >= 2025 THEN yr_score END) - MAX(yr_score), 2) AS change_from_peak
FROM yearly
GROUP BY name, segment
HAVING AVG(CASE WHEN yr >= 2025 THEN yr_score END) IS NOT NULL
ORDER BY change_from_peak
""", engine)

peak_vs_recent

,name,segment,peak,recent,change_from_peak
0,Pepper,digital_bank,4.50,2.16,-2.34
1,Mizrahi-Tefahot,traditional_bank,4.26,2.26,-2.00
2,Mercantile Discount,traditional_bank,4.22,2.42,-1.80
3,Discount Bank,traditional_bank,4.14,2.42,-1.72
4,Max,credit_card,4.67,3.20,-1.47
5,One Zero,digital_bank,3.43,2.33,-1.10
6,Bank Hapoalim,traditional_bank,4.50,3.49,-1.01
7,First International Bank,traditional_bank,4.42,3.81,-0.60
8,Bank Leumi,traditional_bank,3.85,3.71,-0.14
9,Cal,credit_card,4.33,4.24,-0.08
